# Step 8 · def1: do the white-box evasions survive cheap defenses?
`scripts/defense_checks.py` (~15 min on one GPU):
- **A. Fluency filter:** GPT-2 log-perplexity, threshold at 1% / 5% FPR on val negatives.
- **B. Registry-relative calibration** on twins of the attacked DeBERTa and E5 detectors (re-trained with the same seed).
- **C. Combined:** calibrated detector OR fluency filter.

**Settings:** GPU **T4 x1** is enough (it queues until one of the Step 7 runs frees a slot) · Internet **On** ·
Secrets **Hugging Face** + **mcp-git** · **Save Version → Save & Run All**.
No GPU free and you want part A now? Set `ARGS` to the CPU line below and pick Accelerator **None**.


In [ ]:
# ---- settings for this run ----
REPO = "github.com/Arn15/mcp-poison-detect"
BRANCH = "main"
SCRIPT = "scripts/defense_checks.py"
ARGS = "--tag def1"
# ARGS = "--tag def1_cpu --set 'defense.parts=[fluency]'"   # CPU-only: fluency filter only
GPU_DEPS = False
MIN_COMMIT_FILE = "scripts/defense_checks.py"


In [ ]:
import os, subprocess, base64
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
def secret(k):
    try: return sec.get_secret(k)
    except Exception: return None
HF_SECRET, GH_SECRET = "Hugging Face", "mcp-git"   # names of your Kaggle secrets
os.environ["HF_TOKEN"] = secret(HF_SECRET) or ""
gh = secret(GH_SECRET)
assert gh, "Secret 'mcp-git' not attached to this notebook (Add-ons -> Secrets)"
WORK = "/kaggle/working"

# Token goes in an HTTP header for this command only: never in the URL, .git/config, or error output.
b64 = base64.b64encode(f"x-access-token:{gh}".encode()).decode()
auth = ["-c", f"http.extraHeader=Authorization: Basic {b64}"]
def git(*args, cwd=None):
    r = subprocess.run(["git", *auth, *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(gh, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg.strip()}")
    return r.stdout

os.chdir(WORK)
repo_dir = os.path.join(WORK, "mcp-poison-detect")
if os.path.isdir(os.path.join(repo_dir, ".git")):
    git("fetch", "-q", cwd=repo_dir); git("checkout", "-q", BRANCH, cwd=repo_dir); git("pull", "-q", cwd=repo_dir)
else:
    git("clone", "-q", "-b", BRANCH, f"https://{REPO}.git", repo_dir)
os.chdir(repo_dir)
print("commit:", git("log", "-1", "--oneline", cwd=repo_dir))
assert os.path.exists(MIN_COMMIT_FILE), f"{MIN_COMMIT_FILE} missing: push the latest commit first"


In [ ]:
!pip install -q -r requirements.txt
if GPU_DEPS:
    !pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU"
!python {SCRIPT} {ARGS}


In [ ]:
import glob
run = sorted(glob.glob("/kaggle/working/outputs/defense_checks/*"))[-1]
print(run)
print(open(f"{run}/defense_results.md").read())
